# Shirt Sponsor Churn — Phase 1: Building the Dataset

**Business question:** when a whole sector disappears from a sponsorship market, who takes its place, and how long do they stay?

Spain banned betting brands from football shirts from the 2021/22 season. The Premier League's ban starts in 2026/27. LaLiga is the natural experiment that shows what the Premier League is about to go through.

**This notebook builds the panel** that the analysis in Phase 2 needs:

| Output | Grain | Use |
|---|---|---|
| `club_seasons.csv` | club × season × division | Main shirt sponsor, kit maker, final position, promotion/relegation |
| `sponsors_to_tag.csv` | unique sponsor name | Sector, country and canonical brand (filled in manually) |
| `source_revisions.csv` | page | Wikipedia revision ID of every page used, so the dataset can be rebuilt exactly |

**Scope:** LaLiga, Segunda División, Premier League and Championship, 2005/06 → 2026/27. Second tiers are included so that a relegated club's next sponsor is still observed: without them, relegation would look like the sponsorship ending.

**Source:** the "Personnel and kits" and league tables of each season's English Wikipedia page, read through the MediaWiki API.

## 0. Setup
Runs in Colab (data saved to Google Drive) or locally (data saved next to the notebook). Requires `lxml` for `pd.read_html`.

In [ ]:
import re, json, time, unicodedata, difflib
from io import StringIO
from pathlib import Path

import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.width', 180)
pd.set_option('display.max_columns', 30)
pd.set_option('display.max_colwidth', 60)

FORCE_REFRESH = False      # True = download every page again

try:                       # Colab: save to Drive; locally: next to the notebook
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/proyecto_patrocinios')
except ImportError:
    BASE = Path.cwd() / 'proyecto_patrocinios'

RAW = BASE / 'data' / 'raw'
HTML_DIR = RAW / 'html'
PROC = BASE / 'data' / 'processed'
MANUAL = BASE / 'data' / 'manual'
FIG = BASE / 'figures'
for d in (HTML_DIR, PROC, MANUAL, FIG):
    d.mkdir(parents=True, exist_ok=True)
print('Saving under', BASE)

### Scope
Page titles changed over time (e.g. *FA Premier League* until 2006/07, *Football League Championship* until 2015/16), so each division has a list of candidate titles. The API follows redirects, and the first title that exists is used.

In [ ]:
FIRST, LAST = 2005, 2026          # season start years: 2005/06 ... 2026/27

def season_label(y):
    return f'{y}–{str(y + 1)[-2:]}'     # en dash, as in Wikipedia titles

DIVISIONS = {
    # key: (country, tier, display name, candidate title templates)
    'laliga':       ('ESP', 1, 'LaLiga',         ['{s} La Liga']),
    'segunda':      ('ESP', 2, 'Segunda',        ['{s} Segunda División']),
    'premier':      ('ENG', 1, 'Premier League', ['{s} Premier League', '{s} FA Premier League']),
    'championship': ('ENG', 2, 'Championship',   ['{s} EFL Championship', '{s} Football League Championship']),
}
SEASONS = list(range(FIRST, LAST + 1))
CURRENT_SEASON = LAST              # standings for the current season are incomplete

# Regulatory events used in Phase 2 (verify before publishing)
BETTING_BAN = {'ESP': 2021, 'ENG': 2026}   # first season with no betting brands on the front of the shirt
print(f'{len(DIVISIONS)} divisions × {len(SEASONS)} seasons = {len(DIVISIONS) * len(SEASONS)} pages')

## 1. Download the pages
One request per page, cached as HTML, with the revision ID stored. Wikimedia asks automated clients to identify themselves in the User-Agent; a generic project name is enough.

In [ ]:
API = 'https://en.wikipedia.org/w/api.php'
HEADERS = {'User-Agent': 'SponsorChurnResearch/1.0 (data portfolio project; python-requests)'}
session = requests.Session()
session.headers.update(HEADERS)

def fetch_page(title, retries=6):
    params = {'action': 'parse', 'page': title, 'prop': 'text|revid', 'redirects': 1,
              'format': 'json', 'formatversion': 2, 'disablelimitreport': 1, 'maxlag': 5}
    for attempt in range(retries):
        try:
            r = session.get(API, params=params, timeout=30)
        except requests.RequestException as e:
            wait, why = 5 * (attempt + 1), type(e).__name__
        else:
            if r.status_code == 200:
                data = r.json()
                err = data.get('error', {}).get('code')
                if err is None:
                    p = data['parse']
                    return {'title': p['title'], 'revid': p['revid'], 'html': p['text']}
                if err == 'missingtitle':
                    return None                          # page really doesn't exist
                wait, why = int(r.headers.get('Retry-After', 10)), f'API error: {err}'
            else:
                wait = int(r.headers.get('Retry-After', 10 * (attempt + 1)))
                why = f'HTTP {r.status_code}'
        print(f'   {title}: {why} → waiting {wait}s ({attempt + 1}/{retries})')
        time.sleep(wait)
    return 'FAILED'

revisions, failed = [], []
for key, (country, tier, name, templates) in DIVISIONS.items():
    for y in SEASONS:
        path = HTML_DIR / f'{key}_{y}.html'
        meta = HTML_DIR / f'{key}_{y}.json'
        if path.exists() and meta.exists() and not FORCE_REFRESH:
            revisions.append(json.loads(meta.read_text()))
            continue
        page = None
        for tpl in templates:
            page = fetch_page(tpl.format(s=season_label(y)))
            if page:
                break
            time.sleep(1)
        if page == 'FAILED':
            failed.append((key, y))                      # not cached: re-run the cell to retry
            continue
        info = {'division': key, 'season': y, 'title': page['title'] if page else None,
                'revid': page['revid'] if page else None}
        if page:
            path.write_text(page['html'], encoding='utf-8')
        meta.write_text(json.dumps(info))
        revisions.append(info)
        time.sleep(1)                                    # stay well under the API rate limit
    print(f'✓ {name}')

revisions = pd.DataFrame(revisions)
revisions.to_csv(RAW / 'source_revisions.csv', index=False)
print(f'Pages found: {revisions.title.notna().sum()} / {len(revisions)}')
if failed:
    print(f'⚠️ {len(failed)} pages failed (not cached) — run this cell again to retry them:', failed)
missing_pages = revisions[revisions.title.isna()]
if len(missing_pages):
    display(missing_pages)

## 2. Parse the tables

**Kits table:** the table that has a *kit manufacturer* column and a *sponsor* column. The header changes over time ("Shirt sponsor", "Main shirt sponsor", "Main kit sponsor"), so columns are matched by keyword. Columns such as "Other kit sponsor(s)" or sleeve sponsors are ignored: the study covers the **main front-of-shirt sponsor** only.

**League table:** the table with *Pos*, *Team* and *Pts* columns. Markers after the team name give the outcome: (C) champions, (P) promoted, (O) play-off winners, (R) relegated.

In [ ]:
REF_RE = re.compile(r'\[[^\]]*\]')          # footnotes like [a], [12]
MARK_RE = re.compile(r'\(([A-Z](?:,\s*[A-Z])*)\)\s*$')

def flat_cols(df):
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [' '.join(dict.fromkeys(str(c) for c in col if 'Unnamed' not in str(c))).strip()
                      for col in df.columns]
    df.columns = [REF_RE.sub('', str(c)).strip() for c in df.columns]
    return df

def clean_text(x):
    if pd.isna(x):
        return ''
    x = REF_RE.sub('', str(x))
    return re.sub(r'\s+', ' ', x).strip()

def find_col(cols, include, exclude=()):
    for c in cols:
        lc = c.lower()
        if all(k in lc for k in include) and not any(k in lc for k in exclude):
            return c
    return None

MAKER_KEYS = (['kit', 'manufacturer'], ['manufacturer'], ['kit', 'maker'], ['kit', 'supplier'], ['supplier'])

def promote_header(t):
    # some tables put the header in a <td> row, so pandas numbers the columns
    if len(t) > 1 and all(str(c).strip().isdigit() for c in t.columns):
        t = t.copy()
        t.columns = [clean_text(x) for x in t.iloc[0]]
        t = t.iloc[1:].reset_index(drop=True)
    return t

def parse_kits(html):
    best = None
    for t in pd.read_html(StringIO(html)):
        t = promote_header(flat_cols(t))
        cols = list(t.columns)
        team = find_col(cols, ['team']) or find_col(cols, ['club'])
        maker = next((c for k in MAKER_KEYS if (c := find_col(cols, k))), None)
        excl = ('other', 'sleeve', 'back', 'secondary', 'training', 'shorts')
        sponsor = (find_col(cols, ['main', 'sponsor'], excl) or find_col(cols, ['shirt', 'sponsor'], excl)
                   or find_col(cols, ['kit', 'sponsor'], excl) or find_col(cols, ['sponsor'], excl))
        if not (team and sponsor and len(t) >= 10):
            continue
        if not maker and not re.search('shirt|kit', sponsor, re.I):
            continue                               # a bare 'Sponsor' column without kit context is too risky
        sub = t[[team, maker, sponsor]] if maker else t[[team, sponsor]].assign(_m='')[[team, '_m', sponsor]]
        sub = sub.set_axis(['team_raw', 'kit_maker', 'sponsor_raw'], axis=1)
        if best is None or len(sub) > len(best):
            best = sub
    return best

def parse_standings(html):
    for t in pd.read_html(StringIO(html)):
        t = promote_header(flat_cols(t))
        cols = list(t.columns)
        pos, team, pts, pld = (find_col(cols, ['pos']), find_col(cols, ['team']) or find_col(cols, ['club']),
                               find_col(cols, ['pts']), find_col(cols, ['pld']))
        if pos and team and pts and pld and 10 <= len(t) <= 26:
            out = t[[pos, team, pld, pts]].set_axis(['pos', 'team_raw', 'played', 'points'], axis=1)
            out = out[pd.to_numeric(out.pos, errors='coerce').notna()]
            return out
    return None

def split_markers(team):
    team = clean_text(team)
    m = MARK_RE.search(team)
    marks = set(re.split(r',\s*', m.group(1))) if m else set()
    return MARK_RE.sub('', team).strip(), marks

In [ ]:
kit_rows, table_rows, coverage = [], [], []
for _, r in revisions[revisions.title.notna()].iterrows():
    html = (HTML_DIR / f'{r.division}_{r.season}.html').read_text(encoding='utf-8')
    country, tier, name, _ = DIVISIONS[r.division]
    kits = parse_kits(html)
    stand = parse_standings(html)
    coverage.append({'division': r.division, 'season': r.season,
                     'kits_rows': 0 if kits is None else len(kits),
                     'table_rows': 0 if stand is None else len(stand)})
    if kits is not None:
        k = kits.assign(division=r.division, country=country, tier=tier, season=r.season)
        kit_rows.append(k)
    if stand is not None:
        s = stand.copy()
        parsed = s.team_raw.map(split_markers)
        s['team'] = parsed.str[0]
        s['champion'] = parsed.str[1].map(lambda m: 'C' in m)
        s['promoted'] = parsed.str[1].map(lambda m: 'P' in m)
        s['relegated'] = parsed.str[1].map(lambda m: 'R' in m)
        s = s.assign(division=r.division, season=r.season)
        table_rows.append(s.drop(columns='team_raw'))

kits = pd.concat(kit_rows, ignore_index=True)
tables = pd.concat(table_rows, ignore_index=True)
coverage = pd.DataFrame(coverage)
kits.to_csv(RAW / 'kits_raw.csv', index=False)
tables.to_csv(RAW / 'league_tables_raw.csv', index=False)
print(f'Kit rows: {len(kits):,} | league-table rows: {len(tables):,}')

### QA 1 — coverage by division and season
Every division should have 18–24 rows per season. Zeros mean the page has no kits table (common in some early Segunda seasons) or the parser missed it; those seasons are listed so they can be checked by hand.

In [ ]:
cov = coverage.pivot(index='season', columns='division', values='kits_rows').reindex(columns=list(DIVISIONS))
display(cov.style.background_gradient(cmap='RdYlGn', vmin=0, vmax=24).format('{:.0f}', na_rep='—'))
gaps = coverage[(coverage.kits_rows == 0)]
print(f'Seasons with no kits table: {len(gaps)}')

In [ ]:
# QA 1b — what is on the pages where no kits table was found?
for _, g in gaps.iterrows():
    html = (HTML_DIR / f'{g.division}_{g.season}.html').read_text(encoding='utf-8')
    hits = []
    for t in pd.read_html(StringIO(html)):
        t = promote_header(flat_cols(t))
        cols = [str(c) for c in t.columns]
        if any(re.search(r'sponsor|kit|manufactur|supplier', c, re.I) for c in cols):
            hits.append(f'{len(t)} rows: {cols}')
    print(f"{g.division} {g.season} | 'sponsor' in page: {'sponsor' in html.lower()} | {hits[:2] or 'no table with kit/sponsor columns'}")

## 3. Clean team and sponsor names

**Teams:** names must match between the kits table and the league table, and across divisions (a club relegated to Segunda must keep the same name). Mismatches are listed with suggestions; add fixes to `TEAM_ALIASES` and re-run.

**Sponsors:** footnotes are removed and empty cells become `NO SPONSOR`. Cells with more than one sponsor or a mid-season change ("X / Y", "X (until January)") are flagged for review rather than guessed.

In [ ]:
# Fill in after reading the QA output below
TEAM_ALIASES = {
    'Betis': 'Real Betis',
    'Celta de Vigo': 'Celta Vigo',
    'Deportivo': 'Deportivo La Coruña', 'RC Deportivo': 'Deportivo La Coruña', 'Deportivo A Coruña': 'Deportivo La Coruña',
    'Racing de Santander': 'Racing Santander',
    'Sporting': 'Sporting Gijón', 'Sporting de Gijón': 'Sporting Gijón',
    'Recreativo de Huelva': 'Recreativo', 'Recreativo Huelva': 'Recreativo',
    'Huddesfield Town': 'Huddersfield Town',
    'Real Oviedo': 'Oviedo',
}

def norm_team(t):
    t = clean_text(t)
    return TEAM_ALIASES.get(t, t)

NONE_VALUES = {'', '—', '-', '–', 'none', 'n/a', 'no sponsor', 'nan', '–none–'}
MULTI_RE = re.compile(r'\s/\s|\buntil\b|\bfrom\b|\bhome\b|\baway\b|\bcup\b|\bleague\b|;|\n', re.I)

def clean_sponsor(s):
    s = clean_text(s)
    return 'NO SPONSOR' if s.lower() in NONE_VALUES else s

def sponsor_key(s):
    s = unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode().lower()
    s = re.sub(r'\.(com|es|co\.uk|net|org|eu)\b', '', s)
    return re.sub(r'[^a-z0-9]+', '', s)

kits['team'] = kits.team_raw.map(norm_team)
tables['team'] = tables.team.map(norm_team)
kits['sponsor'] = kits.sponsor_raw.map(clean_sponsor)
kits['sponsor_key'] = kits.sponsor.map(sponsor_key)
kits['multi_sponsor_flag'] = kits.sponsor.str.contains(MULTI_RE)
kits['kit_maker'] = kits.kit_maker.map(clean_text)
print(f"No sponsor: {(kits.sponsor == 'NO SPONSOR').sum()} club-seasons | flagged multi/mid-season: {kits.multi_sponsor_flag.sum()}")

In [ ]:
# QA 2 — teams in the kits table that don't match the league table of the same season
m = kits.merge(tables[['division', 'season', 'team']], on=['division', 'season', 'team'], how='left', indicator=True)
unmatched = m[m._merge == 'left_only'][['division', 'season', 'team']]
all_table_teams = sorted(tables.team.unique())
unmatched = unmatched.assign(suggestion=unmatched.team.map(
    lambda t: (difflib.get_close_matches(t, all_table_teams, n=1, cutoff=0.6) or [''])[0]))
print(f'Unmatched club-seasons: {len(unmatched)} of {len(kits)}')
display(unmatched.drop_duplicates(['team', 'suggestion']).head(40))
print('→ If a suggestion is right, add it to TEAM_ALIASES (kits name → table name) and re-run this section.')

In [ ]:
# QA 3 — same club, different spelling across seasons/divisions (would break the club's history)
names = pd.Series(sorted(set(kits.team) | set(tables.team)))
pairs = []
for i, a in enumerate(names):
    for b in names[i + 1:]:
        ra = difflib.SequenceMatcher(None, a.lower(), b.lower()).ratio()
        if ra >= 0.82:
            pairs.append({'name_a': a, 'name_b': b, 'similarity': round(ra, 2)})
pairs = pd.DataFrame(pairs, columns=['name_a', 'name_b', 'similarity']).sort_values('similarity', ascending=False)
print('Possible duplicates (check; real different clubs such as Real Madrid / Real Betis will not reach 0.82):')
display(pairs.head(40))
print(f'{len(pairs)} pairs above 0.82 (showing top 40)')

## 4. Assemble the club-season panel

In [ ]:
panel = kits.merge(tables[['division', 'season', 'team', 'pos', 'played', 'points', 'champion', 'promoted', 'relegated']],
                   on=['division', 'season', 'team'], how='left')
panel['season_label'] = panel.season.map(season_label)
panel['season_complete'] = panel.season < CURRENT_SEASON
panel = panel[['country', 'division', 'tier', 'season', 'season_label', 'team', 'sponsor', 'sponsor_key',
               'multi_sponsor_flag', 'kit_maker', 'pos', 'played', 'points', 'champion', 'promoted',
               'relegated', 'season_complete', 'sponsor_raw', 'team_raw']]
panel = panel.sort_values(['country', 'team', 'season', 'tier'])

# A club listed on several rows in one season (e.g. one row per one-off sponsor): keep the first row
dupes = panel.duplicated(['team', 'season'], keep=False)
if dupes.any():
    print('Club-seasons listed on several rows (first row kept):')
    display(panel[dupes][['team', 'season', 'division', 'sponsor']])
    panel = panel.drop_duplicates(['team', 'season'], keep='first')
print(f'Panel: {len(panel):,} club-seasons | {panel.team.nunique()} clubs')
panel.to_csv(PROC / 'club_seasons.csv', index=False)

## 5. Sponsors to tag

Phase 2 needs three things per sponsor that no table gives: the **canonical brand** (so *Fly Emirates* → *Emirates* counts as a renewal, not a new sponsor), the **sector**, and the **brand's home country** (to compare foreign vs. domestic sponsors).

This cell exports every unique sponsor with context. Send `sponsors_to_tag.csv` back for tagging; the filled file goes to `data/manual/sponsor_tags.csv`.

**Sector list (fixed in advance):** betting & gaming · crypto & trading apps · airline & travel · banking & insurance · real estate & construction · automotive · telecom & tech · food & drink · energy & utilities · retail & consumer goods · regional/public body · club foundation/own brand · other · no sponsor.

In [ ]:
BETTING_HINT = re.compile(r'bet|casino|poker|bwin|888|sportingbet|betway|william ?hill|codere|sportium|'
                          r'stake|marathon|dafabet|12bet|188|w88|fun88|kaiyun|sbotop|leovegas|unibet|betfair', re.I)
CRYPTO_HINT = re.compile(r'crypto|coin|bitci|chiliz|socios|whalefin|bybit|okx|binance|etoro|trading|fx|forex', re.I)

agg = (panel[panel.sponsor != 'NO SPONSOR']
       .groupby('sponsor_key')
       .agg(sponsor=('sponsor', lambda s: s.value_counts().index[0]),
            variants=('sponsor', lambda s: ' | '.join(sorted(set(s)))[:200]),
            club_seasons=('team', 'size'),
            clubs=('team', lambda s: ', '.join(sorted(set(s)))[:200]),
            countries=('country', lambda s: ','.join(sorted(set(s)))),
            first_season=('season', 'min'), last_season=('season', 'max'),
            any_multi_flag=('multi_sponsor_flag', 'any'))
       .reset_index()
       .sort_values('club_seasons', ascending=False))
agg['sector_hint'] = np.select([agg.sponsor.str.contains(BETTING_HINT), agg.sponsor.str.contains(CRYPTO_HINT)],
                               ['betting & gaming?', 'crypto & trading?'], '')
agg['brand'] = ''
agg['sector'] = ''
agg['brand_country'] = ''
agg['notes'] = ''
agg.to_csv(MANUAL / 'sponsors_to_tag.csv', index=False, encoding='utf-8-sig')
print(f'Unique sponsors: {len(agg)} → saved to {MANUAL / "sponsors_to_tag.csv"}')
print(f"Betting hints: {(agg.sector_hint == 'betting & gaming?').sum()} | crypto hints: {(agg.sector_hint == 'crypto & trading?').sum()}")
display(agg.head(15))

## 6. First look (preliminary — before brand merging)

These charts use raw names, so a rename such as *Fly Emirates* → *Emirates* still counts as a change. They are sanity checks, not results.

In [ ]:
RED, BLUE, GREY = '#CB3524', '#2E6DB4', '#A3A3A3'
plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True,
                     'grid.alpha': 0.25, 'figure.dpi': 110, 'legend.frameon': False,
                     'axes.titleweight': 'bold', 'axes.titlelocation': 'left'})

top = panel[panel.tier == 1].copy()
top = top.sort_values(['team', 'season'])
top['prev_key'] = top.groupby('team').sponsor_key.shift()
top['prev_season'] = top.groupby('team').season.shift()
top['changed'] = (top.prev_season == top.season - 1) & (top.sponsor_key != top.prev_key)
top['observed'] = top.prev_season == top.season - 1

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for div, col in [('laliga', RED), ('premier', BLUE)]:
    d = top[top.division == div]
    rate = d[d.observed].groupby('season').changed.mean() * 100
    axes[0].plot(rate.index, rate.values, marker='o', color=col, label=DIVISIONS[div][2])
    nos = d.groupby('season').sponsor.apply(lambda s: (s == 'NO SPONSOR').sum())
    axes[1].plot(nos.index, nos.values, marker='o', color=col, label=DIVISIONS[div][2])
for ax in axes:
    for c, y in BETTING_BAN.items():
        ax.axvline(y, color=RED if c == 'ESP' else BLUE, ls='--', lw=0.9, alpha=0.6)
axes[0].set_title('Clubs changing main shirt sponsor (% of clubs, raw names)')
axes[1].set_title('Top-flight clubs with no shirt sponsor')
axes[0].legend()
fig.tight_layout()
fig.savefig(FIG / 'p1_preliminary_churn.png', dpi=160, bbox_inches='tight')
plt.show()

In [ ]:
# Quick check of the press figure for LaLiga 2026/27 (main sponsor only)
d = top[(top.division == 'laliga') & (top.season == CURRENT_SEASON)]
print(f"LaLiga {season_label(CURRENT_SEASON)}: {d.changed.sum()} of {d.observed.sum()} clubs that were already in LaLiga changed main sponsor (raw names).")
print('Promoted clubs are not counted as changes. The press figure (21 exits / 8 entries) likely includes sleeve and other shirt spaces.')

## 7. Inventory

| File | Content |
|---|---|
| `data/raw/source_revisions.csv` | Page title and Wikipedia revision ID per division-season |
| `data/raw/html/` | Cached page HTML |
| `data/raw/kits_raw.csv`, `league_tables_raw.csv` | Parsed tables before cleaning |
| `data/processed/club_seasons.csv` | **The panel** |
| `data/manual/sponsors_to_tag.csv` | Unique sponsors to tag (send back) |

**Next:** tag the sponsors, then Phase 2: churn rates, survival analysis by sector (Kaplan–Meier, Cox), relegation effect and the betting-ban natural experiment.

In [ ]:
for p in sorted(BASE.glob('data/**/*.csv')):
    print(f'{str(p.relative_to(BASE)):55s} {sum(1 for _ in open(p, encoding="utf-8-sig")) - 1:>7,} rows')